# AI CCTV Anomaly Detection - Colab workflow
Dataset inspection → training → threshold calibration → evaluation → inference.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Upload the project zip when asked.
Everything you report must come from *your* runs of these cells. Cells marked SYNTHETIC are software tests only.

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
print(out if out else "No GPU detected: Runtime > Change runtime type > T4 GPU (CPU will work but is slow).")

## 1. Get the project and install dependencies

In [ ]:
import os, zipfile
from google.colab import files
if not os.path.isdir("ai-cctv-anomaly-detection"):
    up = files.upload()                       # choose ai-cctv-anomaly-detection.zip
    zipfile.ZipFile(list(up)[0]).extractall(".")
%cd ai-cctv-anomaly-detection
!pip install -q -r requirements.txt

If pip reported a NumPy/OpenCV binary mismatch, use **Runtime → Restart session**, then re-run from the `%cd` line.

## 2. Software check (SYNTHETIC fixture - not a result)

In [ ]:
!python -m pytest -q tests
!python -m src.train --synthetic
!python -m src.calibrate_threshold --synthetic
!python -m src.evaluate --synthetic

## 3. Download the Kaggle dataset
Upload your `kaggle.json` (Kaggle → Settings → Create New Token). Or upload/unzip the dataset manually into `data/raw/cctv-exam-monitor-dataset/`.

In [ ]:
from google.colab import files
files.upload()                                 # choose kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
!kaggle datasets download -d cctvdataset/cctv-exam-monitor-dataset -p data/raw/cctv-exam-monitor-dataset --unzip

## 4. Inspect the dataset (do this before configuring labels)

In [ ]:
!python -m src.dataset_inspector

## 5. Configure labels from what the inspector showed
Edit `config.yaml` in the left file browser, **or** use the helper below. Examples:
* folders named normal/abnormal → `label_mode='folders'`
* labels CSV → `label_mode='csv'`, `labels_csv='path/to/labels.csv'`
* no labels → `label_mode='none'` (normal-only training; no accuracy/F1 will be computed)

The helper rewrites config.yaml (comments are lost; a backup `config.yaml.bak` is kept).

In [ ]:
import yaml, shutil
shutil.copy("config.yaml", "config.yaml.bak")
cfg = yaml.safe_load(open("config.yaml"))

# >>> EDIT these after reading the inspector output <<<
cfg["data"]["label_mode"] = "auto"                 # auto | folders | csv | none
cfg["data"]["normal_class_names"] = ["normal"]
cfg["data"]["abnormal_class_names"] = ["abnormal", "anomaly", "anomalous"]
cfg["data"]["labels_csv"] = None
cfg["data"]["assume_unlabeled_normal"] = True      # only if unlabeled footage is mostly normal activity
cfg["training"]["epochs"] = 30

yaml.safe_dump(cfg, open("config.yaml", "w"), sort_keys=False)
print("saved")

## 6. Quick real-data pipeline check (tiny), then full training

In [ ]:
!python -m src.train --smoke

In [ ]:
!python -m src.train

In [ ]:
import pandas as pd, IPython.display as d
d.display(pd.read_csv("outputs/training_history.csv").tail())
d.display(d.Image("outputs/training_curve.png")); d.display(d.Image("outputs/reconstruction_samples.png"))

## 7. Calibrate the threshold (validation split only)

In [ ]:
!python -m src.calibrate_threshold
import IPython.display as d, json
print(json.dumps(json.load(open("models/threshold.json")), indent=2))
d.display(d.Image("outputs/threshold_calibration.png"))

## 8. Evaluate on the held-out test split

In [ ]:
!python -m src.evaluate
import IPython.display as d, json, os
m = json.load(open("outputs/evaluation/metrics.json"))
print(m.get("note", "")); print({k: m[k] for k in ["accuracy","precision","recall","f1","roc_auc","pr_auc"] if k in m})
for f in ["confusion_matrix.png","roc_pr_curves.png","score_distribution.png","highest_scoring_frames.png"]:
    p = "outputs/evaluation/" + f
    if os.path.exists(p): d.display(d.Image(p))

## 9. Run inference on a video
Set `VIDEO` to a file from the dataset or upload your own.

In [ ]:
import glob, IPython.display as d, pandas as pd
vids = sorted(glob.glob("data/raw/cctv-exam-monitor-dataset/**/*.mp4", recursive=True) +
              glob.glob("data/raw/cctv-exam-monitor-dataset/**/*.avi", recursive=True))
VIDEO = vids[0] if vids else None            # >>> or set a path yourself <<<
print("Using:", VIDEO)
!python -m src.video_processor --video "$VIDEO"
import os
out = "outputs/inference/" + os.path.splitext(os.path.basename(VIDEO))[0]
d.display(pd.read_csv(out + "/event_log.csv"))
for p in sorted(glob.glob(out + "/evidence/*.jpg"))[:3]: d.display(d.Image(p))
d.display(d.Video(out + "/annotated_video.mp4", embed=True, width=720))

## 10. Save results to your computer
Then copy `models/` to your Mac to run the dashboard: `streamlit run app.py`.

In [ ]:
!zip -qr results.zip models outputs config.yaml
from google.colab import files
files.download("results.zip")